# SuperTrend Entry/Exit on SPY
## Strategy Brief
The SuperTrend indicator is a popular trend-following tool that helps identify the prevailing market trend. It generates buy and sell signals based on price movements and volatility. In this strategy, we apply the SuperTrend indicator to SPY, the ETF that tracks the S&P 500, to determine entry and exit points. The strategy aims to capture significant trends in SPY, potentially improving returns over a simple buy-and-hold approach.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters for the SuperTrend strategy, including the period for the ATR calculation and the multiplier used to determine the SuperTrend bands.

In [ ]:
ATR_PERIOD = 10
ATR_MULTIPLIER = 3.0
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
TICKER = 'SPY'

### PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance, calculate the SuperTrend indicator, and plot it over the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Calculate ATR
high_low = data['High'] - data['Low']
high_close = np.abs(data['High'] - data['Close'].shift())
low_close = np.abs(data['Low'] - data['Close'].shift())
true_range = np.maximum(high_low, high_close, low_close)
atr = true_range.rolling(ATR_PERIOD).mean()

# Calculate SuperTrend
supertrend = pd.DataFrame(index=data.index)
upper_band = ((data['High'] + data['Low']) / 2) + (ATR_MULTIPLIER * atr)
lower_band = ((data['High'] + data['Low']) / 2) - (ATR_MULTIPLIER * atr)

# Initialize SuperTrend
supertrend['SuperTrend'] = np.nan
supertrend['Direction'] = 1

for i in range(1, len(data)):
    if data['Close'][i] > upper_band[i-1]:
        supertrend['SuperTrend'][i] = lower_band[i]
        supertrend['Direction'][i] = 1
    elif data['Close'][i] < lower_band[i-1]:
        supertrend['SuperTrend'][i] = upper_band[i]
        supertrend['Direction'][i] = -1
    else:
        supertrend['SuperTrend'][i] = supertrend['SuperTrend'][i-1]
        supertrend['Direction'][i] = supertrend['Direction'][i-1]

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close', alpha=0.5)
plt.plot(supertrend['SuperTrend'], label='SuperTrend', color='red')
plt.title('SPY Price and SuperTrend')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
In this phase, we define the entry and exit logic based on the SuperTrend signals and create a positions series to represent our trading positions.

In [ ]:
# Signal: 1 for buy, -1 for sell
signals = supertrend['Direction']

# Entry/Exit logic
positions = signals.shift(1)
positions.fillna(0, inplace=True)

### PHASE 4 - Coding & Backtesting
We will calculate daily returns based on the positions and plot the resulting equity curve to visualize the strategy's performance.

In [ ]:
# Calculate daily returns
returns = data['Close'].pct_change()
strategy_returns = returns * positions

# Equity curve
(1 + strategy_returns).cumprod().plot(figsize=(14, 7), title='Equity Curve')
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy using key performance metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown. We will also compare these metrics to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (1 + returns).prod() ** (252 / len(returns)) - 1
    sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252)
    downside_std = returns[returns < 0].std()
    sortino_ratio = returns.mean() / downside_std * np.sqrt(252)
    max_drawdown = (1 + returns).cumprod().div((1 + returns).cumprod().cummax()).min() - 1
    calmar_ratio = cagr / abs(max_drawdown)
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

strategy_metrics = calculate_performance_metrics(strategy_returns.dropna())
buy_and_hold_metrics = calculate_performance_metrics(returns.dropna())

# Comparison table
metrics_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': strategy_metrics,
    'Buy & Hold': buy_and_hold_metrics
})
print(metrics_df)

### PHASE 6 - Deploy & Monitor
We will create a function that downloads the last 60 days of SPY data, computes today's SuperTrend signal, and prints the current trading position.

In [ ]:
def get_current_position():
    recent_data = yf.download(TICKER, period='60d')
    
    # Recalculate ATR and SuperTrend for recent data
    high_low = recent_data['High'] - recent_data['Low']
    high_close = np.abs(recent_data['High'] - recent_data['Close'].shift())
    low_close = np.abs(recent_data['Low'] - recent_data['Close'].shift())
    true_range = np.maximum(high_low, high_close, low_close)
    atr = true_range.rolling(ATR_PERIOD).mean()
    
    upper_band = ((recent_data['High'] + recent_data['Low']) / 2) + (ATR_MULTIPLIER * atr)
    lower_band = ((recent_data['High'] + recent_data['Low']) / 2) - (ATR_MULTIPLIER * atr)
    
    supertrend = pd.DataFrame(index=recent_data.index)
    supertrend['SuperTrend'] = np.nan
    supertrend['Direction'] = 1

    for i in range(1, len(recent_data)):
        if recent_data['Close'][i] > upper_band[i-1]:
            supertrend['SuperTrend'][i] = lower_band[i]
            supertrend['Direction'][i] = 1
        elif recent_data['Close'][i] < lower_band[i-1]:
            supertrend['SuperTrend'][i] = upper_band[i]
            supertrend['Direction'][i] = -1
        else:
            supertrend['SuperTrend'][i] = supertrend['SuperTrend'][i-1]
            supertrend['Direction'][i] = supertrend['Direction'][i-1]

    current_position = supertrend['Direction'].iloc[-1]
    print(f"Current position based on SuperTrend: {'Long' if current_position == 1 else 'Short'}")

get_current_position()